In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start=None):
    """Find repo root by walking upward until expected project markers are found."""
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for path in [current, *current.parents]:
        if (path / "config").exists() and (path / "notebooks").exists() and (path / "src").exists():
            return path
    raise RuntimeError("Could not find repo root. Launch notebook from inside the repository root.")

REPO_ROOT = find_repo_root()

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

PREPROCESSING_DATA_DIR = REPO_ROOT / "data" / "processed" / "preprocessing"
OBJ2_DATA_DIR = REPO_ROOT / "data" / "processed" / "objective2_demand"
OBJ2_OUTPUT_DIR = REPO_ROOT / "outputs" / "objective2_demand"
OBJ2_MODEL_DIR = OBJ2_OUTPUT_DIR / "models"
OBJ2_VALIDATION_DIR = OBJ2_OUTPUT_DIR / "validation"
CONFIG_DIR = REPO_ROOT / "config"
CALENDAR_DIR = CONFIG_DIR / "calendars"
LOOKUP_DIR = CONFIG_DIR / "lookups"

for path in [OBJ2_DATA_DIR, OBJ2_OUTPUT_DIR, OBJ2_MODEL_DIR, OBJ2_VALIDATION_DIR]:
    path.mkdir(parents=True, exist_ok=True)


In [ ]:
# Task 8 input/output paths
FES_PATH = PREPROCESSING_DATA_DIR / "fes_demand_annual_2030_2045.parquet"
FUTURE_DEMAND_HOURLY_RAW_PATH = OBJ2_DATA_DIR / "future_demand_hourly_raw_2030_2045.parquet"
DEMAND_FUTURE_HOURLY_PATH = OBJ2_DATA_DIR / "demand_future_hourly_2030_2045.parquet"
DEMAND_FUTURE_QA_SUMMARY_PATH = OBJ2_VALIDATION_DIR / "demand_future_hourly_2030_2045_QA_summary.csv"
DEMAND_FUTURE_README_PATH = OBJ2_VALIDATION_DIR / "demand_future_hourly_2030_2045_README.md"


In [ ]:
hourly_final = pd.read_parquet(
    FUTURE_DEMAND_HOURLY_RAW_PATH
).copy()

hourly_final["timestamp_utc"] = pd.to_datetime(hourly_final["timestamp_utc"], utc=True)

required_final_cols = [
    "timestamp_utc",
    "year",
    "fes_scenario",
    "climate_member",
    "demand_mw",
]

missing_final_cols = sorted(set(required_final_cols) - set(hourly_final.columns))
assert not missing_final_cols, f"Missing final output columns: {missing_final_cols}"

hourly_final = hourly_final[required_final_cols].copy()

# Expected hours for 2030-2045 inclusive.
expected_hours = pd.date_range(
    "2030-01-01 00:00:00",
    "2045-12-31 23:00:00",
    freq="h",
    tz="UTC",
)

actual_hours = pd.Index(hourly_final["timestamp_utc"].drop_duplicates().sort_values())

missing_hours = expected_hours.difference(actual_hours)
extra_hours = actual_hours.difference(expected_hours)

assert len(missing_hours) == 0, f"Missing timestamps: {len(missing_hours)}"
assert len(extra_hours) == 0, f"Unexpected timestamps: {len(extra_hours)}"

dup_count = hourly_final.duplicated(
    ["timestamp_utc", "fes_scenario", "climate_member"]
).sum()

assert dup_count == 0, f"Duplicate hourly keys: {dup_count}"

assert hourly_final["demand_mw"].notna().all()
assert (hourly_final["demand_mw"] > 0).all()

n_members = hourly_final["climate_member"].nunique()
n_scenarios = hourly_final["fes_scenario"].nunique()
expected_rows = len(expected_hours) * n_members * n_scenarios

print("Expected rows:", expected_rows)
print("Actual rows:", len(hourly_final))

assert len(hourly_final) == expected_rows


In [ ]:
# Annual reconciliation

fes = pd.read_parquet(FES_PATH).copy()
fes = fes[fes["fes_scenario"].isin(["Holistic Transition", "Electric Engagement"])].copy()
fes = fes[(fes["year"] >= 2030) & (fes["year"] <= 2045)].copy()

annual_hourly = (
    hourly_final
    .groupby(["year", "fes_scenario", "climate_member"], as_index=False)
    ["demand_mw"]
    .sum()
)

annual_hourly["hourly_annual_twh"] = annual_hourly["demand_mw"] / 1_000_000

annual_recon = annual_hourly.merge(
    fes[["year", "fes_scenario", "annual_demand_twh"]],
    on=["year", "fes_scenario"],
    how="left",
)

annual_recon["diff_twh"] = (
    annual_recon["hourly_annual_twh"] - annual_recon["annual_demand_twh"]
)

print(annual_recon["diff_twh"].abs().describe())

assert annual_recon["diff_twh"].abs().max() < 1e-6

In [ ]:
# Save final output

hourly_final.to_parquet(
    DEMAND_FUTURE_HOURLY_PATH,
    index=False,
)

qa_summary = {
    "start_timestamp_utc": str(hourly_final["timestamp_utc"].min()),
    "end_timestamp_utc": str(hourly_final["timestamp_utc"].max()),
    "n_rows": int(len(hourly_final)),
    "n_scenarios": int(n_scenarios),
    "n_climate_members": int(n_members),
    "duplicate_keys": int(dup_count),
    "max_abs_annual_reconciliation_diff_twh": float(annual_recon["diff_twh"].abs().max()),
}

pd.Series(qa_summary).to_csv(DEMAND_FUTURE_QA_SUMMARY_PATH)


In [ ]:
final_readme = """# demand_future_hourly_2030_2045_README.md

Purpose:
Final Objective 2 future hourly GB demand output for 2030-2045.

Output:
demand_future_hourly_2030_2045.parquet

Columns:
- timestamp_utc: hourly UTC timestamp
- year: calendar year
- fes_scenario: FES ED1 scenario, restricted to Holistic Transition and Electric Engagement
- climate_member: selected UKCP18 member
- demand_mw: hourly average GB National Demand in MW

Method:
1. Generated climate-only daily demand using the selected Task 3 XGBoost model.
2. Filtered to selected UKCP18 climate members only.
3. Anchored each year to FES ED1 annual demand totals.
4. Disaggregated daily MWh to hourly MW using the historic ND hourly shape library.
5. Reconciled hourly totals back to daily anchored demand and annual FES demand.

Units:
- Daily demand: MWh
- Annual FES demand: TWh, converted to MWh for scaling
- Hourly demand: MW, numerically equal to hourly MWh because each row is one hour

Boundary:
Great Britain only: England, Wales, and Scotland. Northern Ireland is excluded.

Core demand definition:
NESO National Demand.
"""

(DEMAND_FUTURE_README_PATH).write_text(final_readme)
